# AI in Dentistry Workshop
## Orange → Colab: Interactive Diabetes Machine-Learning Lab

### Workshop learning path
**Understand the Code → Build Without Code → Interpret Like a Healthcare Researcher**

This notebook follows the same logic as the Orange workflow:

`File → Data Table → Feature Statistics → Rank → Models → Test & Score → Tree Viewer → Predictions`

You will build:

1. **Classification** — predict whether a patient belongs to class `0` or `1`.
2. **Regression** — predict a numerical glucose value.
3. **Deployment simulation** — test the finished model on **20 patients kept completely unseen during model development**.

### How to use this notebook

🟦 **RUN THIS** — code is already prepared. Run the cell.

🟨 **YOUR TURN** — change one simple value, rerun, and observe the effect.

🟩 **DISCUSS** — stop and interpret the result before continuing.

> You do **not** need to write Python from scratch. The goal is to understand what Orange is doing under the hood.

## Orange workflow ↔ Python workflow

| Orange | Colab |
|---|---|
| File | `pandas.read_csv()` |
| Data Table | `display(df)` |
| Feature Statistics | `describe()` + data-quality checks |
| Rank | feature-importance calculation |
| Logistic Regression | `LogisticRegression()` |
| Decision Tree | `DecisionTreeClassifier()` / `DecisionTreeRegressor()` |
| Random Forest | `RandomForestClassifier()` / `RandomForestRegressor()` |
| Linear Regression | `LinearRegression()` |
| Test & Score | cross-validation + metrics |
| Tree Viewer | `plot_tree()` |
| Predictions | `.predict()` |

## 🟦 RUN THIS — 1. Install and import the required libraries

In [ ]:
!pip -q install kagglehub

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from math import log2
from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, KFold,
    cross_validate, cross_val_predict
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, make_scorer,
    ConfusionMatrixDisplay, RocCurveDisplay,
    mean_squared_error, mean_absolute_error, r2_score
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

print("Ready.")

# PART A — LOAD THE DATASET

## 🟦 RUN THIS — 2. Download the Pima Indians Diabetes dataset from Kaggle

This is the same type of dataset used in the Orange demonstration.

The notebook automatically downloads the dataset and identifies the CSV file.

In [ ]:
path = kagglehub.dataset_download("kumargh/pimaindiansdiabetescsv")

csv_files = [
    os.path.join(root, file)
    for root, _, files_in_folder in os.walk(path)
    for file in files_in_folder
    if file.lower().endswith(".csv")
]

if not csv_files:
    raise FileNotFoundError("No CSV file was found in the Kaggle dataset.")

csv_path = csv_files[0]
df = pd.read_csv(csv_path)

# Some copies of this dataset may load without headers.
expected_cols = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
]

if list(df.columns) != expected_cols and df.shape[1] == 9:
    # If the first row was mistaken for a header, reload without header.
    test_reload = pd.read_csv(csv_path, header=None)
    if test_reload.shape[1] == 9:
        test_reload.columns = expected_cols
        df = test_reload

print("Dataset file:", csv_path)
print("Full dataset shape:", df.shape)
display(df.head())

## 🟩 DISCUSS

Before training anything:

- How many patients are in the dataset?
- How many variables are there?
- Which column is the classification target?
- Which variable will later become the regression target?

For this workshop:

- **Classification target:** `Outcome`
- **Regression target:** `Glucose`

# PART B — CREATE THE 20-PATIENT DEPLOYMENT SIMULATION

## 🟦 RUN THIS — 3. Reserve 20 patients before model development

These **20 cases are not a normal training/validation split**.

They simulate patients arriving **after the model has already been developed and frozen**.

They must not be used for:

- feature ranking,
- model comparison,
- cross-validation,
- hyperparameter decisions,
- threshold selection,
- or model fitting.

The remaining cases form the **development dataset**.

In [ ]:
development_df, deployment20_df = train_test_split(
    df,
    test_size=20,
    random_state=42,
    stratify=df["Outcome"]
)

development_df = development_df.reset_index(drop=True)
deployment20_df = deployment20_df.reset_index(drop=True)

print("Full dataset:", len(df))
print("Development dataset:", len(development_df))
print("Deployment simulation:", len(deployment20_df))

print("\nClass distribution in the 20 deployment cases:")
display(
    deployment20_df["Outcome"]
    .value_counts()
    .sort_index()
    .rename("Number of patients")
    .to_frame()
)

### Why use `stratify=Outcome`?

The 20 deployment cases retain approximately the same proportion of class `0` and class `1` as the full dataset.

### Why use `random_state=42`?

So everyone in the workshop gets the **same 20 patients** when the notebook is rerun.

> **Workshop wording:** call these **“20 unseen deployment-simulation patients”**, not a definitive clinical validation cohort.

## 🟦 RUN THIS — 4. Create blind deployment files

In [ ]:
# Classification blind set: hide Outcome, but keep all predictors including Glucose
classification_blind20 = deployment20_df.drop(columns=["Outcome"]).copy()
classification_blind20.to_csv(
    "deployment_20_classification_blind.csv",
    index=False
)

# Regression blind set: hide Glucose.
# Outcome is also excluded here because our primary regression experiment will not use it.
regression_blind20 = deployment20_df.drop(columns=["Glucose", "Outcome"]).copy()
regression_blind20.to_csv(
    "deployment_20_regression_blind.csv",
    index=False
)

# Instructor/reference version
deployment20_df.to_csv(
    "deployment_20_reference_answers.csv",
    index=False
)

print("Created:")
print("• deployment_20_classification_blind.csv")
print("• deployment_20_regression_blind.csv")
print("• deployment_20_reference_answers.csv")

# PART C — DATA EXPLORATION

## 🟦 RUN THIS — 5. Orange `Data Table`

From now on, exploration is performed on the **development dataset**, not the 20 deployment patients.

In [ ]:
display(development_df.head(10))

print("Development dataset shape:", development_df.shape)
print("\nColumns:")
print(list(development_df.columns))

## 🟦 RUN THIS — 6. Orange `Feature Statistics`

In [ ]:
stats = pd.DataFrame({
    "Mean": development_df.mean(numeric_only=True),
    "Median": development_df.median(numeric_only=True),
    "Minimum": development_df.min(numeric_only=True),
    "Maximum": development_df.max(numeric_only=True),
    "Missing": development_df.isna().sum(),
    "Missing %": development_df.isna().mean() * 100
})

display(stats)

## 🟦 RUN THIS — 7. Look for clinically suspicious zero values

Orange may say there are no empty cells, but a numerical value of `0` can still be clinically implausible for some measurements.

In [ ]:
zero_check_cols = [
    c for c in ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
    if c in development_df.columns
]

zero_summary = pd.DataFrame({
    "Zero count": [(development_df[c] == 0).sum() for c in zero_check_cols],
    "Zero %": [(development_df[c] == 0).mean() * 100 for c in zero_check_cols]
}, index=zero_check_cols)

display(zero_summary)

## 🟩 DISCUSS — Data quality

Ask the group:

> **If Orange reports “Missing = 0”, does that prove the data are clean?**

No.

A stored `0` is not the same thing as an empty cell. Clinical data need **clinical interpretation**, not only software checks.

For the core Orange-to-code comparison, we keep the original values unchanged. An optional cleaning section appears later.

# PART 1 — CLASSIFICATION

## Question

> **Which class does this patient belong to — 0 or 1?**

For this dataset:

- `Outcome = 0`
- `Outcome = 1`

The task is therefore **binary classification**.

## 🟦 RUN THIS — 8. Separate predictors and target

In [ ]:
X_class = development_df.drop(columns=["Outcome"])
y_class = development_df["Outcome"].astype(int)

print("Predictors:")
print(list(X_class.columns))

print("\nClass distribution:")
display(y_class.value_counts().sort_index().rename("Patients").to_frame())

## 🟦 RUN THIS — 9. Orange `Rank`

We reproduce the concept of feature ranking using:

- approximate **Gain Ratio**
- tree-based **Gini importance**

Exact values may differ from Orange because continuous-variable discretization may differ.

In [ ]:
def entropy(values):
    probs = pd.Series(values).value_counts(normalize=True)
    return -sum(p * log2(p) for p in probs if p > 0)

def approximate_gain_ratio(X, y, bins=5):
    result = {}
    base_entropy = entropy(y)

    for col in X.columns:
        try:
            grouped_feature = pd.qcut(X[col], q=bins, duplicates="drop")
        except Exception:
            grouped_feature = pd.cut(X[col], bins=bins, duplicates="drop")

        temp = pd.DataFrame({
            "feature_bin": grouped_feature,
            "target": y
        }).dropna()

        conditional_entropy = 0
        split_information = 0

        for _, group in temp.groupby("feature_bin", observed=False):
            p = len(group) / len(temp)
            conditional_entropy += p * entropy(group["target"])
            if p > 0:
                split_information -= p * log2(p)

        information_gain = base_entropy - conditional_entropy
        result[col] = (
            information_gain / split_information
            if split_information > 0 else 0
        )

    return pd.Series(result, name="Approx_Gain_Ratio")

gain_ratio = approximate_gain_ratio(X_class, y_class)

rank_tree = DecisionTreeClassifier(random_state=42)
rank_tree.fit(X_class, y_class)

gini_importance = pd.Series(
    rank_tree.feature_importances_,
    index=X_class.columns,
    name="Gini_Importance"
)

rank_table = pd.concat(
    [gain_ratio, gini_importance],
    axis=1
).sort_values("Approx_Gain_Ratio", ascending=False)

display(rank_table)

## 🟩 DISCUSS — Feature importance

- Which variable ranks highest?
- Does “most important feature” mean “this feature causes diabetes”?

> **No. Predictive importance is not causation.**

## 🟨 YOUR TURN — 10. Choose simple model settings

Change **only one value at a time**, rerun, and later compare performance.

Suggested experiments:

- `TREE_MAX_DEPTH = 3`, then `5`, then `None`
- `RF_TREES = 50`, then `100`, then `300`

Do not hunt for the highest score using the 20 deployment cases. Those remain hidden until the end.

In [ ]:
TREE_MAX_DEPTH = None     # Try: 3, 5, None
RF_TREES = 300            # Try: 50, 100, 300

print("Decision Tree max depth:", TREE_MAX_DEPTH)
print("Random Forest number of trees:", RF_TREES)

## 🟦 RUN THIS — 11. Build the three classification models

In [ ]:
classification_models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=TREE_MAX_DEPTH,
        random_state=42
    ),

    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=RF_TREES,
        random_state=42
    )
}

print("Models created.")

## 🟦 RUN THIS — 12. Orange `Test & Score`

We use **20-fold stratified cross-validation** on the development dataset.

Each fold becomes the test fold once while the other folds are used for training.

In [ ]:
cv_class = StratifiedKFold(
    n_splits=20,
    shuffle=True,
    random_state=42
)

classification_scoring = {
    "AUC": "roc_auc",
    "Accuracy": "accuracy",
    "F1": "f1",
    "Precision": "precision",
    "Recall": "recall",
    "MCC": make_scorer(matthews_corrcoef)
}

classification_rows = []

for model_name, model in classification_models.items():
    scores = cross_validate(
        model,
        X_class,
        y_class,
        cv=cv_class,
        scoring=classification_scoring,
        n_jobs=-1
    )

    row = {"Model": model_name}
    for metric in classification_scoring:
        row[metric] = scores[f"test_{metric}"].mean()

    classification_rows.append(row)

classification_results = (
    pd.DataFrame(classification_rows)
    .sort_values("AUC", ascending=False)
    .reset_index(drop=True)
)

display(
    classification_results.style.format({
        "AUC": "{:.3f}",
        "Accuracy": "{:.3f}",
        "F1": "{:.3f}",
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "MCC": "{:.3f}"
    })
)

## 🟩 DISCUSS — Which classification model would you choose?

Do not look only at Accuracy.

Ask:

- Which has the highest AUC?
- Which has the highest Recall?
- Which has the highest Precision?
- Which has the strongest MCC?
- In a screening problem, what would a false negative mean?
- If false alarms trigger expensive procedures, why might Precision matter?

> **The best metric depends on the consequence of the error.**

## 🟦 RUN THIS — 13. Confusion matrix and ROC curve for Logistic Regression

In [ ]:
logreg_demo = classification_models["Logistic Regression"]

y_pred_cv = cross_val_predict(
    logreg_demo,
    X_class,
    y_class,
    cv=cv_class,
    method="predict",
    n_jobs=-1
)

y_prob_cv = cross_val_predict(
    logreg_demo,
    X_class,
    y_class,
    cv=cv_class,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

print("Accuracy:", round(accuracy_score(y_class, y_pred_cv), 3))
print("Recall:", round(recall_score(y_class, y_pred_cv), 3))
print("Precision:", round(precision_score(y_class, y_pred_cv), 3))
print("F1:", round(f1_score(y_class, y_pred_cv), 3))
print("AUC:", round(roc_auc_score(y_class, y_prob_cv), 3))
print("MCC:", round(matthews_corrcoef(y_class, y_pred_cv), 3))

ConfusionMatrixDisplay.from_predictions(y_class, y_pred_cv)
plt.title("Cross-validated Confusion Matrix — Logistic Regression")
plt.show()

RocCurveDisplay.from_predictions(y_class, y_prob_cv)
plt.title("Cross-validated ROC Curve — Logistic Regression")
plt.show()

## 🟦 RUN THIS — 14. Orange `Tree Viewer`

For teaching, we show only the first 3 levels so the tree remains readable.

In [ ]:
tree_view = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)
tree_view.fit(X_class, y_class)

plt.figure(figsize=(18, 8))
plot_tree(
    tree_view,
    feature_names=X_class.columns,
    class_names=["0", "1"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Classification Tree — First 3 Levels")
plt.show()

## 🟩 DISCUSS — Tree thresholds

A threshold such as `Glucose > X` is a **data-derived model split**.

It is **not automatically a clinical diagnostic cut-off**.

## 🟦 RUN THIS — 15. Choose the best classification model from development data and freeze it

The choice is made **before the 20 deployment cases are evaluated**.

For this notebook, the model with the highest mean cross-validated **AUC** is selected automatically.

In [ ]:
best_class_name = classification_results.iloc[0]["Model"]
final_class_model = clone(classification_models[best_class_name])

# Train the chosen final model on ALL development cases
final_class_model.fit(X_class, y_class)

print("Selected final classification model:", best_class_name)
print("Final model trained on all", len(development_df), "development patients.")
print("The model is now FROZEN for the deployment simulation.")

# CLASSIFICATION DEPLOYMENT SIMULATION — 20 NEW PATIENTS

## 🟦 RUN THIS — 16. Let the frozen model predict the 20 unseen patients

At this point, pretend these are **new patients arriving after deployment**.

The true `Outcome` is intentionally hidden from the prediction table.

In [ ]:
X_deploy_class = classification_blind20[X_class.columns].copy()

deployment_class_predictions = classification_blind20.copy()
deployment_class_predictions["AI_Prediction"] = final_class_model.predict(X_deploy_class)

if hasattr(final_class_model, "predict_proba"):
    deployment_class_predictions["P(Outcome=1)"] = (
        final_class_model.predict_proba(X_deploy_class)[:, 1]
    )

display(deployment_class_predictions)

## 🟨 YOUR TURN — Before revealing the answers

Look at the 20 predictions.

1. How many cases do you think the model got right?
2. Which predictions look uncertain?
3. Does a probability close to `0.50` feel as convincing as `0.95`?
4. Would you make a clinical decision from the AI prediction alone?

**Do not run the next cell until the instructor says “Reveal.”**

## 🟦 REVEAL — 17. Compare the AI predictions with the real Outcome

In [ ]:
y_deploy_class = deployment20_df["Outcome"].astype(int)
pred_deploy_class = final_class_model.predict(X_deploy_class)

deployment_class_results = deployment_class_predictions.copy()
deployment_class_results["True_Outcome"] = y_deploy_class.values
deployment_class_results["Correct"] = (
    deployment_class_results["AI_Prediction"]
    == deployment_class_results["True_Outcome"]
)

display(deployment_class_results)

print("\n20-case deployment simulation:")
print("Correct:",
      int(deployment_class_results["Correct"].sum()),
      "out of",
      len(deployment_class_results))
print("Accuracy:",
      f'{accuracy_score(y_deploy_class, pred_deploy_class):.1%}')
print("Precision:",
      f'{precision_score(y_deploy_class, pred_deploy_class, zero_division=0):.3f}')
print("Recall:",
      f'{recall_score(y_deploy_class, pred_deploy_class, zero_division=0):.3f}')
print("F1:",
      f'{f1_score(y_deploy_class, pred_deploy_class, zero_division=0):.3f}')
print("MCC:",
      f'{matthews_corrcoef(y_deploy_class, pred_deploy_class):.3f}')

if hasattr(final_class_model, "predict_proba"):
    prob_deploy_class = final_class_model.predict_proba(X_deploy_class)[:, 1]
    print("AUC:",
          f'{roc_auc_score(y_deploy_class, prob_deploy_class):.3f}')

## 🟩 DISCUSS — What did deployment teach us?

The 20 cases are useful for demonstrating what happens when a finished model encounters new patients.

But:

> **20 patients are too few to provide a precise standalone estimate of clinical performance.**

That is why the larger development-set cross-validation remains important.

# PART 2 — REGRESSION

## Question

> **What numerical glucose value do we predict?**

Now the target is `Glucose`.

This is **regression**, because the output is a number.

## 🟩 DISCUSS — Should `Outcome` be included as a feature?

This depends on the real use case.

### Exclude it
If `Outcome` was derived from the same glucose value or would not be known at prediction time.

### Potentially include it
Only if it truly means a **prior established diagnosis** that existed before the glucose value being predicted.

> **Rule:** ask whether the information was available at prediction time and whether it was derived from the target.

For the primary workshop model, we exclude `Outcome`.

## 🟦 RUN THIS — 18. Prepare the regression dataset

In [ ]:
X_reg = development_df.drop(columns=["Glucose", "Outcome"])
y_reg = development_df["Glucose"].astype(float)

print("Regression target: Glucose")
print("Predictors:")
print(list(X_reg.columns))

## 🟨 YOUR TURN — 19. Choose simple regression model settings

Change one parameter and observe whether model performance changes.

In [ ]:
REG_TREE_MAX_DEPTH = None   # Try: 3, 5, None
REG_RF_TREES = 300          # Try: 50, 100, 300

print("Regression Tree max depth:", REG_TREE_MAX_DEPTH)
print("Random Forest trees:", REG_RF_TREES)

## 🟦 RUN THIS — 20. Build regression models

In [ ]:
regression_models = {
    "Regression Tree": DecisionTreeRegressor(
        max_depth=REG_TREE_MAX_DEPTH,
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=REG_RF_TREES,
        random_state=42
    ),

    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])
}

print("Regression models created.")

## 🟦 RUN THIS — 21. Orange `Test & Score` for regression

We use **10-fold cross-validation**.

For regression:

- lower **MSE, RMSE, MAE** = better
- higher **R²** = better

In [ ]:
cv_reg = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

regression_rows = []

for model_name, model in regression_models.items():
    cv_pred = cross_val_predict(
        model,
        X_reg,
        y_reg,
        cv=cv_reg,
        n_jobs=-1
    )

    mse = mean_squared_error(y_reg, cv_pred)

    regression_rows.append({
        "Model": model_name,
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_reg, cv_pred),
        "R²": r2_score(y_reg, cv_pred)
    })

regression_results = (
    pd.DataFrame(regression_rows)
    .sort_values(["RMSE", "MAE"])
    .reset_index(drop=True)
)

display(
    regression_results.style.format({
        "MSE": "{:.3f}",
        "RMSE": "{:.3f}",
        "MAE": "{:.3f}",
        "R²": "{:.3f}"
    })
)

## 🟩 DISCUSS — Regression metrics

### MAE
Average absolute error in the original unit.

If MAE = 21, the model is wrong by about **21 mg/dL on average**.

### RMSE
Also in mg/dL, but larger errors are penalized more.

### MSE
Squared error. Useful mathematically but less intuitive clinically.

### R²
How much of the variation in glucose is explained by the model.

A negative R² means the model performs worse than simply predicting the average glucose value.

### MAPE
Do not emphasize MAPE here because zero glucose values can make percentage error explode.

## 🟦 RUN THIS — 22. Regression `Tree Viewer`

In [ ]:
reg_tree_view = DecisionTreeRegressor(
    max_depth=3,
    random_state=42
)
reg_tree_view.fit(X_reg, y_reg)

plt.figure(figsize=(18, 8))
plot_tree(
    reg_tree_view,
    feature_names=X_reg.columns,
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Regression Tree — First 3 Levels")
plt.show()

## 🟨 YOUR TURN — 23. Leakage / prior-information experiment

Try both settings:

```python
USE_PRIOR_DIABETES_STATUS = False
```

then:

```python
USE_PRIOR_DIABETES_STATUS = True
```

Only interpret the second scenario as valid if `Outcome` genuinely represents a **prior-known diagnosis**.

In [ ]:
USE_PRIOR_DIABETES_STATUS = False

if USE_PRIOR_DIABETES_STATUS:
    X_reg_experiment = development_df.drop(columns=["Glucose"])
    explanation = "Outcome INCLUDED — valid only if it is truly prior-known."
else:
    X_reg_experiment = development_df.drop(columns=["Glucose", "Outcome"])
    explanation = "Outcome EXCLUDED."

print(explanation)

experiment_models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42
    ),
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])
}

experiment_rows = []

for model_name, model in experiment_models.items():
    pred = cross_val_predict(
        model,
        X_reg_experiment,
        y_reg,
        cv=cv_reg,
        n_jobs=-1
    )

    mse = mean_squared_error(y_reg, pred)

    experiment_rows.append({
        "Model": model_name,
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_reg, pred),
        "R²": r2_score(y_reg, pred)
    })

display(pd.DataFrame(experiment_rows).style.format({
    "RMSE": "{:.3f}",
    "MAE": "{:.3f}",
    "R²": "{:.3f}"
}))

## 🟩 DISCUSS — The key leakage question

If including `Outcome` improves performance, ask:

> **Is that improvement legitimate predictive information, or is the model indirectly receiving information derived from the target?**

Better performance is not automatically better science.

## 🟦 RUN THIS — 24. Choose and freeze the final regression model

The model with the lowest cross-validated **RMSE** is selected from the primary regression experiment where `Outcome` was excluded.

In [ ]:
best_reg_name = regression_results.iloc[0]["Model"]
final_reg_model = clone(regression_models[best_reg_name])

final_reg_model.fit(X_reg, y_reg)

print("Selected final regression model:", best_reg_name)
print("Final regression model trained on all development patients.")
print("The model is now FROZEN.")

# REGRESSION DEPLOYMENT SIMULATION — THE SAME 20 NEW PATIENTS

## 🟦 RUN THIS — 25. Predict glucose without seeing the true glucose values

In [ ]:
X_deploy_reg = regression_blind20[X_reg.columns].copy()

deployment_reg_predictions = regression_blind20.copy()
deployment_reg_predictions["Predicted_Glucose"] = final_reg_model.predict(X_deploy_reg)

display(deployment_reg_predictions)

## 🟨 YOUR TURN — Before revealing actual glucose

Look at the predicted glucose values.

Ask:

1. Which predictions would you expect to be closest?
2. Which cases might be difficult?
3. What size error would be clinically meaningful?
4. Does a numerical prediction automatically mean the model is clinically useful?

**Do not run the next cell until the instructor says “Reveal.”**

## 🟦 REVEAL — 26. Compare predicted vs actual glucose

In [ ]:
actual_glucose = deployment20_df["Glucose"].astype(float)
predicted_glucose = final_reg_model.predict(X_deploy_reg)

deployment_reg_results = deployment_reg_predictions.copy()
deployment_reg_results["Actual_Glucose"] = actual_glucose.values
deployment_reg_results["Absolute_Error"] = np.abs(
    deployment_reg_results["Predicted_Glucose"]
    - deployment_reg_results["Actual_Glucose"]
)

display(deployment_reg_results)

deploy_mse = mean_squared_error(actual_glucose, predicted_glucose)

print("\n20-case regression deployment simulation:")
print("MAE:", round(mean_absolute_error(actual_glucose, predicted_glucose), 3), "mg/dL")
print("RMSE:", round(np.sqrt(deploy_mse), 3), "mg/dL")
print("MSE:", round(deploy_mse, 3))
print("R²:", round(r2_score(actual_glucose, predicted_glucose), 3))

# OPTIONAL ADVANCED SECTION — CLEANING ZERO VALUES SAFELY

This is separate from the core Orange reproduction.

If the dataset documentation and clinical context establish that zero is invalid for a given variable, it can be converted to missing and imputed.

The important principle is:

> **Preprocessing should be learned inside the training process, not from the deployment patients.**

In [ ]:
clean_dev = development_df.copy()

possible_invalid_zero_cols = [
    c for c in ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
    if c in clean_dev.columns
]

for col in possible_invalid_zero_cols:
    clean_dev[col] = clean_dev[col].replace(0, np.nan)

X_clean = clean_dev.drop(columns=["Outcome"])
y_clean = clean_dev["Outcome"].astype(int)

clean_logreg = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

clean_scores = cross_validate(
    clean_logreg,
    X_clean,
    y_clean,
    cv=cv_class,
    scoring=classification_scoring,
    n_jobs=-1
)

print("Cleaned-data Logistic Regression:")
for metric in classification_scoring:
    print(
        metric,
        round(clean_scores[f"test_{metric}"].mean(), 3)
    )

# 🟦 RUN THIS — 27. Export the workshop results

This creates CSV files that candidates can download from Colab.

In [ ]:
deployment_class_results.to_csv(
    "deployment_classification_results_20.csv",
    index=False
)

deployment_reg_results.to_csv(
    "deployment_regression_results_20.csv",
    index=False
)

classification_results.to_csv(
    "classification_cross_validation_results.csv",
    index=False
)

regression_results.to_csv(
    "regression_cross_validation_results.csv",
    index=False
)

print("Saved:")
print("• deployment_classification_results_20.csv")
print("• deployment_regression_results_20.csv")
print("• classification_cross_validation_results.csv")
print("• regression_cross_validation_results.csv")

# FINAL WORKSHOP SUMMARY

## Classification
**Question:** Which class does the patient belong to?

`Development data → Rank → Logistic Regression / Tree / Random Forest → Cross-validation → Choose model → Freeze → 20 unseen patients`

Main metrics:

**AUC + Accuracy + Precision + Recall + F1 + MCC**

---

## Regression
**Question:** What numerical glucose value do we predict?

`Development data → Regression Tree / Random Forest / Linear Regression → Cross-validation → Choose model → Freeze → 20 unseen patients`

Main metrics:

**MAE + RMSE + MSE + R²**

---

## What candidates should remember

1. **Classification predicts a label; regression predicts a number.**
2. **Do not let deployment/test cases influence model development.**
3. **Feature importance does not mean causation.**
4. **“No missing cells” does not guarantee clean clinical data.**
5. **The best evaluation metric depends on the clinical consequence of error.**
6. **Information available only after the outcome cannot be used as a predictor.**
7. **Higher performance caused by leakage is not better science.**
8. **A frozen model must be tested on genuinely unseen patients.**
9. **Twenty deployment cases are a teaching demonstration, not sufficient evidence of clinical readiness.**
10. **A working AI model still requires robust validation before clinical use.**

### Final learning journey

**See the code → Run it → Interpret it → Change one thing → Observe the effect → Build the equivalent workflow in Orange → Simulate deployment**